# AI Agent Using Tools
### CampusX concepts consolidated into one practical Jupyter notebook

This notebook compresses the core learning path into one runnable workflow:

**LLM → Tools → Tool Schema → Tool Binding → Tool Call → Tool Execution → ToolMessage → Final Answer → Agent Loop**

The key idea:

> **AI Agent = LLM (reasoning/decision making) + Tools (actions)**

An LLM does not magically execute Python functions or APIs. It decides that a tool should be called and produces structured arguments. The application executes the tool, sends the result back to the model, and repeats until the model can answer.

**Source notebooks supplied by the learner**
- https://colab.research.google.com/drive/1pat55z_iiLqzInsLi3sWS2wekFCXprQW
- https://colab.research.google.com/drive/1GHHGsDFB5266Cc0xDsZ6OWzkB5GGSxFW
- https://colab.research.google.com/drive/1O7cdBtiP_GNXgL9Iz4LPzYfvTKtMtv25

> Note: the first URL was supplied twice, so it is intentionally included only once.


## 1. What is a Tool?

A **tool is a function that gives an LLM an external capability**.

Examples:
- calculator
- web search
- database query
- weather API
- code interpreter
- company/internal API

Without tools:

`User → LLM → Text`

With tools:

`User → LLM → Tool Call → Tool → Tool Result → LLM → Final Answer`

### Built-in vs Custom Tools

| Type | Meaning | Example |
|---|---|---|
| Built-in | Pre-made by a library/provider | Search tool |
| Custom | Function written by you | Currency converter |
| Toolkit | Group of related tools | Database toolkit |

A good tool needs a clear **name, description, and input schema**. The description matters because it helps the model decide when the tool is appropriate.


In [ ]:
# Install the main packages.
# Run this cell once in a fresh Colab/Jupyter environment.

%pip install -q langchain-core langchain-openai python-dotenv

In [ ]:
import os
from getpass import getpass

# Do NOT hard-code API keys into notebooks that you share.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage


## 2. Create Custom Tools

LangChain's `@tool` decorator converts an ordinary Python function into a tool with a schema the LLM can inspect.

The **docstring is important**. It becomes part of the tool description.


In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression such as '125 * 8' or '(50 + 10) / 2'."""
    try:
        # Educational example only. Do not expose unrestricted eval() to untrusted users.
        allowed = set("0123456789+-*/(). %")
        if not set(expression) <= allowed:
            return "Error: expression contains unsupported characters."
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


@tool
def word_count(text: str) -> str:
    """Count the number of whitespace-separated words in a text."""
    return str(len(text.split()))


tools = [calculator, word_count]

for t in tools:
    print("Name:", t.name)
    print("Description:", t.description)
    print("Schema:", t.args_schema.model_json_schema())
    print("-" * 70)


## 3. Tool Calling

Tool calling has two separate stages:

1. **The LLM chooses a tool and creates arguments.**
2. **Our application executes the tool.**

This distinction is fundamental. The model proposes the action; your program remains responsible for actually performing it.

### Message types

- `HumanMessage`: user request
- `AIMessage`: model response, possibly containing `tool_calls`
- `ToolMessage`: result returned by the executed tool
- `SystemMessage`: system-level instructions


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Bind tools so the model knows what capabilities are available.
llm_with_tools = llm.bind_tools(tools)

messages = [
    HumanMessage(content="Calculate 125 * 8 and tell me the result.")
]

ai_msg = llm_with_tools.invoke(messages)

print("Model text:", ai_msg.content)
print("Tool calls:", ai_msg.tool_calls)


In [ ]:
# Execute the tool calls produced by the model.
# The model has requested an action; our Python application performs it.

for call in ai_msg.tool_calls:
    selected_tool = {t.name: t for t in tools}[call["name"]]
    result = selected_tool.invoke(call["args"])
    print(f"Tool: {call['name']}")
    print(f"Arguments: {call['args']}")
    print(f"Result: {result}")


## 4. The Complete Tool-Calling Loop

The previous cells showed the pieces separately. Now we combine them:

**User**
↓  
**LLM decides whether a tool is needed**
↓  
**AIMessage with tool call**
↓  
**Application executes the tool**
↓  
**ToolMessage contains the result**
↓  
**LLM sees the result**
↓  
**Final answer**

This is the foundation from which an autonomous agent is built.


In [ ]:
def run_tool_calling_agent(user_query: str, max_steps: int = 5):
    """Minimal agent loop demonstrating LLM + tools without hiding the mechanics."""

    messages = [HumanMessage(content=user_query)]

    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # No tool call means the model has reached its final answer.
        if not response.tool_calls:
            return response.content

        # Execute every requested tool call.
        for call in response.tool_calls:
            selected_tool = {t.name: t for t in tools}.get(call["name"])

            if selected_tool is None:
                result = f"Unknown tool: {call['name']}"
            else:
                try:
                    result = selected_tool.invoke(call["args"])
                except Exception as e:
                    result = f"Tool error: {e}"

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"],
                )
            )

    return "Stopped: maximum agent steps reached."


print(run_tool_calling_agent(
    "What is (125 * 8) + (50 / 2)? Use the calculator tool."
))


## 5. Why This Becomes an Agent

A normal chain usually follows a fixed path:

`A → B → C → D`

An agent can choose its next action dynamically:

`Goal → Think/Decide → Tool → Observe → Decide again → ... → Answer`

This is commonly described as **ReAct: Reasoning + Acting**.

### Example

User:

> What is the capital of France and what is its population?

An agent may:

1. Recognize that current population information requires external information.
2. Select a search tool.
3. Execute the search.
4. Inspect the result.
5. Decide whether another lookup is needed.
6. Produce the final answer.

The important part is not the word “autonomous”. The important part is the **feedback loop**:

**decide → act → observe → decide again**.


## 6. Adding a Real External Tool

A production agent normally connects tools to external systems.

Typical architecture:

```text
                 ┌───────────────┐
                 │      LLM      │
                 │ reasoning     │
                 └───────┬───────┘
                         │
                    tool call
                         ↓
              ┌────────────────────┐
              │    Tool Router     │
              └─────────┬──────────┘
                        │
          ┌─────────────┼─────────────┐
          ↓             ↓             ↓
      Calculator     Web Search    Database
          │             │             │
          └─────────────┼─────────────┘
                        ↓
                  Tool result
                        ↓
                       LLM
                        ↓
                  Final answer
```

For a real web-search agent, you can plug in a search provider such as Tavily, DuckDuckGo, or another API. The exact provider is replaceable because the agent only needs a well-defined tool interface.


## 7. Tool Schema and Validation

The LLM needs to know:

- tool name
- what the tool does
- which arguments it accepts
- the type of each argument
- which arguments are required

For more complex tools, use a **Pydantic schema**.

Example concept:

```python
class SearchInput(BaseModel):
    query: str
    max_results: int = 5
```

Then the model can produce structured arguments such as:

```json
{
  "query": "latest AI agent frameworks",
  "max_results": 5
}
```

Schema validation is especially useful when tools perform real actions such as database writes, purchases, emails, or API operations.


## 8. Tool Errors and Agent Recovery

Real tools fail.

Possible failures:
- API timeout
- invalid input
- missing permissions
- rate limit
- malformed response
- unavailable service

A robust agent should return useful errors to the model instead of crashing the entire workflow.

A practical pattern is:

```python
try:
    result = tool.invoke(args)
except Exception as e:
    result = f"Tool failed: {e}"
```

The model can then decide whether to retry, choose another tool, or explain the failure.

For sensitive actions, add **human approval** before execution.


## 9. Agent vs Tool Calling

| Tool Calling | Agent |
|---|---|
| Model can request a tool | Model can repeatedly choose actions |
| Application executes the request | Application runs an iterative loop |
| Often one/few steps | Designed for multi-step tasks |
| Developer controls workflow | Model has more decision-making freedom |
| Easier to test | More flexible but harder to control |

So the progression is:

**Functions → Tools → Tool Calling → Tool Loop → Agent**

That is the conceptual backbone of the CampusX material consolidated here.


## 10. A Small Multi-Tool Agent Demo

This final demo gives the model two different capabilities. The model decides which one is appropriate.

Try requests such as:

- `Calculate 17 * 24`
- `How many words are in: artificial intelligence changes software development`
- `Calculate 10% of 850 and explain the answer`


In [ ]:
queries = [
    "Calculate 17 * 24.",
    "How many words are in: artificial intelligence changes software development?",
    "Calculate 10% of 850 and explain the result."
]

for q in queries:
    print("USER:", q)
    print("AGENT:", run_tool_calling_agent(q))
    print("=" * 80)


# Final Mental Model

```text
                 AI AGENT
                    │
          ┌─────────┴─────────┐
          │                   │
         LLM                TOOLS
      reasoning              actions
          │                   │
          └─────────┬─────────┘
                    ↓
              TOOL CALL
                    ↓
              TOOL EXECUTION
                    ↓
               OBSERVATION
                    ↓
              LLM RE-EVALUATES
                    ↓
          ┌─────────┴─────────┐
          │                   │
       another tool        final answer
          │
          └──────→ loop
```

### The six things to remember

1. **Tools extend an LLM beyond text generation.**
2. **A tool is usually a function with a clear schema and description.**
3. **Binding tools tells the LLM what capabilities are available.**
4. **A tool call is a request from the model, not execution by the model.**
5. **The application executes the tool and returns a `ToolMessage`.**
6. **An agent is the repeated decision/action/observation loop around the LLM and tools.**

### One-line summary

> **An LLM provides the brain; tools provide the hands; the agent loop connects the two.**


## Optional Extensions

To turn this educational notebook into a more realistic project, add:

- Web search tool
- Wikipedia tool
- SQL/database tool
- Calculator tool
- Weather API
- RAG retriever as a tool
- Human approval for sensitive actions
- Memory/state
- Streaming
- Tracing and evaluation
- Tool authentication and permission checks
- Retry and timeout policies

A useful next project is a **Research Agent** with:

`User Query → Search Tool → RAG/Document Tool → Calculator → Final Answer`

That combines the CampusX themes of retrieval, tools, tool calling, and agents into one application.
